# YOLO PCX Notebook

## 1) Project paths and environment

In [1]:
import sys
import os

project_root = os.path.abspath("/home/heydari/paper-camera-ready/paper/paper/12-supp")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

## 2) Imports

In [2]:
# === Standard Library ===
import os
import sys
import copy
import math
import logging
import h5py

# === Scientific Computing ===
import numpy as np
import cv2
from sklearn.mixture import GaussianMixture

# === Torch & TorchVision ===
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes, make_grid

# === PIL & Plotting ===
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

# === Progress Bar ===
from tqdm import tqdm

# === CRP & Zennit ===
import zennit.image as zimage
from crp.image import imgify
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names
from yolov6.data.data_augment import letterbox

# === LCRP Utilities ===
from LCRP.models import get_model
from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, VISUALIZATIONS, COMPOSITES

# === Local Project Modules ===
sys.path.append("..")
from src.glocal_analysis import run_analysis
from src.datasets.person_car_dataset import PersonCarDataset
from src.yolo_pcx_test import plot_pcx_explanations
from src.letterbox_utils import letterbox_transform, check_img_size, rescale_boxes

# === Silence noisy loggers ===
os.environ["NUMBA_LOG_LEVEL"] = "WARNING"
os.environ.pop("NUMBA_DEBUG", None)
os.environ.pop("MPLDEBUG", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

for name in ("numba", "numba.core", "numba.core.ssa", "matplotlib",
             "matplotlib.font_manager", "PIL.PngImagePlugin", "PIL.TiffImagePlugin",
             "PIL", "tensorflow"):
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)
    lg.propagate = False
    for h in list(lg.handlers):
        lg.removeHandler(h)
    lg.addHandler(logging.NullHandler())

try:
    mpl.set_loglevel("warning")
except Exception:
    pass


/home/heydari/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'vis_opaque_img_border_v2' from 'LCRP.utils.render' (/home/heydari/paper-camera-ready/paper/paper/LCRP/utils/render.py)

## ============ CONFIGURATION ============

In [ ]:
data_dir = f"/home/jovyan/FHHI-XAI/data/BRK_trial"
ckpt_path = f"/home/jovyan/FHHI-XAI/models/best_ckpt_original.pt"
ref_imgs_path=f"/home/jovyan/FHHI-XAI/output_BRK/ref_imgs/"
output_dir_crp = f"/home/jovyan/FHHI-XAI/output_BRK/crp/yolo_person_car/"
output_dir_pcx=f"/home/jovyan/FHHI-XAI/output_BRK/pcx/yolo_person_car"

n_prototypes_by_layer = {
    "module.backbone.stem.rbr_dense.conv":            {0: 3, 1: 4},
    "module.backbone.ERBlock_2.0.rbr_dense.conv":     {0: 3, 1: 4}
    }

## 3) Dataset loading

In [ ]:
# ============ CREATE DATASETS ============
from functools import partial

# Load datasets with letterbox transform
transform_batch = partial(letterbox_transform, target_size=640, stride=64, half=False, auto=False)
dataset = PersonCarDataset(root_dir=data_dir, split="train", transform=transform_batch)
orig_dataset = PersonCarDataset(root_dir=data_dir, split="train", transform=None)

## 4) Model loading

In [ ]:
model_name = "yolov6s6"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

dtype = torch.float32

model = get_model(model_name=model_name, classes=2, ckpt_path=ckpt_path, device=device, dtype=dtype)
model.to(device);

## 6) PCX explanations visualization

In [ ]:
layer_names = list(n_prototypes_by_layer.keys())
use_half = False

# ============ SPECIFY VALIDATION INDEX ============
sample_idx = 258
prediction_num = None  # None => use the highest-confidence detection

model = model.to(device)
stride = int(model.stride.max()) if hasattr(model, 'stride') else 64
img_size = check_img_size(640, stride=stride)

print(f"Image size: {img_size}, stride: {stride}")
print(f"Processing {len(layer_names)} layers")

# ============ PREPARE INPUT ============
model.eval()

orig_img_raw, label = orig_dataset[sample_idx]

if torch.is_tensor(orig_img_raw):
    if orig_img_raw.dtype == torch.uint8:
        orig_img = Image.fromarray(orig_img_raw.permute(1, 2, 0).numpy())
    else:
        orig_img = Image.fromarray((orig_img_raw.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
elif isinstance(orig_img_raw, np.ndarray):
    orig_img = Image.fromarray(orig_img_raw)
else:
    orig_img = orig_img_raw

orig_np = np.array(orig_img)
original_shape = orig_np.shape[:2]

img_letterbox = letterbox(orig_np, new_shape=img_size, stride=stride, auto=True)[0]
letterbox_shape = img_letterbox.shape[:2]

img_tensor = torch.from_numpy(np.ascontiguousarray(img_letterbox.transpose((2, 0, 1)))).float() / 255.0
img_tensor = img_tensor.to(device)

# Run prediction
with torch.no_grad():
    scores, boxes = model.predict_with_boxes(img_tensor.unsqueeze(0))

boxes_original = rescale_boxes(
    boxes[0].cpu().detach().numpy(),
    letterbox_shape=letterbox_shape,
    original_shape=original_shape
)

num_boxes = boxes_original.shape[0]
class_ids = scores[0].argmax(dim=1)
confidences = scores[0].max(dim=1).values

if num_boxes == 0:
    raise ValueError(f"No detections found for sample_idx={sample_idx}")

if prediction_num is None:
    prediction_num = int(confidences.argmax().item())
elif prediction_num < 0 or prediction_num >= num_boxes:
    raise IndexError(
        f"prediction_num={prediction_num} is out of range for sample_idx={sample_idx}; "
        f"valid range is [0, {num_boxes - 1}]"
    )

print(f"Found {num_boxes} detections")
print(f"Using detection index: {prediction_num}")

if num_boxes > 0:
    for layer_idx, layer_name in enumerate(layer_names):
        print(f"\nLAYER {layer_idx}/{len(layer_names)-1}: {layer_name}")

        prototype_dict = n_prototypes_by_layer[layer_name]
        class_id = class_ids[prediction_num].item()
        conf = confidences[prediction_num].item()

        if class_id not in prototype_dict:
            print(f"  class={class_id} not in prototype_dict, skipping")
            continue

        print(f"  Detection {prediction_num}: class={class_id}, conf={conf:.3f}")

        try:
            img_tensor_cpu = img_tensor.cpu()

            fig = plot_pcx_explanations(
                model_name=model_name,
                model=model,
                img=img_tensor_cpu,
                orig_img=orig_img,
                dataset=dataset,
                orig_dataset=orig_dataset,
                class_id=class_id,
                n_concepts=3,
                n_refimgs=12,
                num_prototypes=prototype_dict,
                prediction_num=prediction_num,
                layer_name=layer_name,
                ref_imgs_path=ref_imgs_path,
                output_dir_pcx=output_dir_pcx,
                output_dir_crp=output_dir_crp,
                letterbox_shape=letterbox_shape,
                original_shape=original_shape,
                rescale_boxes_fn=rescale_boxes,
                dataset_type="BRK"
            )

            model = model.to(device)

            if fig is not None:
                plt.show()
            else:
                print("  No figure returned")

        except Exception as e:
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
            model = model.to(device)
            continue

print("\nDone.")